In [1]:
import yaml 
from pathlib import Path

import mlflow

from pytorch_pipeline import val
from pytorch_pipeline.train import build_pipeline_dataloaders, build_datasets, get_device
from pytorch_pipeline.utils import resolve_uri, Config, resolve_hardware_profile, get_current_git_branch
from pytorch_pipeline.utils.params import DatasetParams, DataLoadersParams, PathsParams

In [23]:
#Args
test = True
test_fraction = 0.1
seed = 42
model_name = 'cv_pheno_inat'
model_version = 1
config_path = Path("/home/etienne/projects/inat-phenology-cv/configs/local.yaml")

In [11]:
# Set up environment specific configs
with open(config_path, "r") as file:
    env_configs = yaml.safe_load(file)
paths_params = PathsParams(**env_configs["paths"])
dataloader_params = DataLoadersParams(**env_configs["dataloader_params"])
hardware_profile = resolve_hardware_profile()
configs = Config(
    config_path,
    paths_params=paths_params,
    dataloaders_params=dataloader_params,
    hardware_profile=hardware_profile,
    git_branch=get_current_git_branch(),
    )

In [14]:
# Load the model
model_uri = f"models:/{model_name}/{model_version}"

# Load the native PyTorch model
model = mlflow.pytorch.load_model(model_uri)

In [24]:
#Load model, dataset & dataloaders 
mlflow.set_tracking_uri(resolve_uri())

configs.test = test
device = get_device()
dataset_params = DatasetParams(testing_frac=test_fraction)
configs.dataset_params = dataset_params
datasets = build_datasets(configs, model, seed= seed)
_, val_loader, _ = build_pipeline_dataloaders(datasets, configs.dataloaders_params, seed=seed)

Running on cuda
Test mode - keeping 10.0% of dataset
1524 observations kept


In [25]:
#Run inference
obs_ids, raw_labels, raw_preds = val.execute(model=model, dataloader=val_loader, device=device, as_numpy=True)

In [30]:
from cleanlab.filter import find_label_issues
import numpy as np

means = []
issues = []

for i in range(3):
    # Format predicted probs for cleanlab
    labels = raw_labels[:,i].astype(int)
    pred_probs_pos = raw_preds[:,i]
    pred_probs_neg = 1 - pred_probs_pos
    pred_probs = np.column_stack((pred_probs_neg, pred_probs_pos))
    issue_mask = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    )
    means.append(issue_mask.mean())

    ordered_issue_indices = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
    )
    
    issues.append([obs_ids[i] for i in ordered_issue_indices])

In [28]:
print(f"Flowering {means[0]}")
print(f"Fruiting {means[1]}")
print(f"Flower_Budding {means[2]}")

Flowering 0.0392156862745098
Fruiting 0.0
Flower_Budding 0.0718954248366013


In [31]:
print(issues)

[[181551770, 287611870, 91867925, 284034684, 91476153, 47732812], [], [218989486, 268485337, 137853747, 121955011, 151353506, 257139230, 25176546, 79174888, 285306322, 126498794, 315097948]]
